# Diabetes Guard - Llama 7B QLoRA Backend (Colab)

## This notebook runs in Google Colab with GPU
- Loads Llama 7B with 4-bit quantization
- Fine-tunes LoRA adapters on recipe data
- Serves as API for React frontend

In [ ]:
# =========================================================
# STEP 1: INSTALL DEPENDENCIES
# =========================================================
!pip install transformers accelerate peft bitsandbytes datasets
!pip install gradio pyngrok

In [ ]:
# =========================================================
# STEP 2: IMPORTS
# =========================================================
import os
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import gradio as gr
from pyngrok import ngrok

In [ ]:
# =========================================================
# STEP 3: LOAD LLAMA 7B WITH 4-BIT QUANTIZATION
# =========================================================
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load base Llama 7B model
# Note: You need to accept the license on HuggingFace first
model_name = "meta-llama/Llama-2-7b-hf"

print("Loading Llama 7B with 4-bit quantization...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Llama 7B loaded successfully!")

In [ ]:
# =========================================================
# STEP 4: SETUP LORA ADAPTERS
# =========================================================
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,  # LoRA rank
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA adapters configured!")

In [ ]:
# =========================================================
# STEP 5: PREPARE TRAINING DATA
# =========================================================
# Load recipe dataset
dataset = load_dataset("ziq/ingredient_to_sugar_level")

# Format data for Llama training (instruction format)
def format_prompt(example):
    return {
        "text": f"""<human>: Is this recipe safe for diabetics? Recipe: {example['text']}\n<bot>: {example['text']} contains approximately {example['labels']}g sugar per serving. {'Not safe for diabetics.' if example['labels'] > 20 else 'Generally safe for diabetics.'}"""
    }

# Apply formatting
dataset = dataset.map(format_prompt)
train_data = dataset["train"].select(range(1000))  # Use 1000 samples for demo
print(f"Training data: {len(train_data)} samples")

In [ ]:
# =========================================================
# STEP 6: TRAIN LORA ADAPTERS (UNCOMMENT TO TRAIN)
# =========================================================
# training_args = TrainingArguments(
#     output_dir="./llama-qlora",
#     num_train_epochs=3,
#     per_device_train_batch_size=4,
#     learning_rate=2e-4,
#     logging_steps=10,
#     save_strategy="steps",
#     save_steps=100,
# )
# 
# trainer = Trainer(
#     model=model,
#     train_dataset=train_data,
#     args=training_args,
#     data_collator=lambda data: tokenizer(
#         return_tensors="pt",
#         text_field=data["text"]
#     )
# )
# 
# # Train (this takes ~30 min on A100)
# trainer.train()
# 
# # Save model
# model.save_pretrained("./diabetes-llama-qlora")
# print("✅ Model trained and saved!")

In [ ]:
# =========================================================
# STEP 7: INFERENCE FUNCTION
# =========================================================
def generate_explanation(recipe_text: str, sugar_grams: float = 20.0, is_safe: bool = False) -> str:
    """Generate natural language explanation"""
    safety = "NOT SAFE for diabetics" if not is_safe else "generally SAFE for diabetics"
    
    prompt = f"""<human>: Explain this recipe for diabetics. Recipe: {recipe_text}, Sugar: {sugar_grams}g per serving, Status: {safety}\n<bot>:"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the response
    if "<bot>:" in response:
        response = response.split("<bot>:")[-1].strip()
    
    return response

# Test
test_result = generate_explanation("2 cups sugar, 1 cup butter", 45.0, False)
print(f"Generated: {test_result}")

In [ ]:
# =========================================================
# STEP 8: GRADIO API (FOR REACT FRONTEND)
# =========================================================
# Set ngrok auth token (you need to get from ngrok.com)
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

# Create Gradio interface
def api_explanation(recipe: str, sugar: float, is_safe: bool):
    return generate_explanation(recipe, float(sugar), bool(is_safe))

interface = gr.Interface(
    fn=api_explanation,
    inputs=[
        gr.Textbox(label="Recipe Ingredients"),
        gr.Slider(0, 100, label="Sugar (grams)"),
        gr.Checkbox(label="Diabetic Safe?")
    ],
    outputs=gr.Textbox(label="Explanation"),
    title="Diabetes Guard - Llama 7B QLoRA",
    description="AI-powered recipe analysis for diabetics"
)

# Launch with ngrok
# public_url = ngrok.connect(7860). publicly available URL
# interface.launch(server_port=7860)

# OR just launch locally (no ngrok)
interface.launch()
print("Gradio API is running!")
# Access at: the URL shown above (e.g., http://localhost:7860)
# For public access, use ngrok or deploy to HuggingFace Spaces